### Models

### Import

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


### Load and Clean Data

In [3]:
# Load dataset
df = pd.read_csv("Traffic_dataset_cleaned.csv")

# Remove leakage and irrelevant columns
cols_to_drop = [
    "CRASH_DATE",
    "MOST_SEVERE_INJURY",
    "INTERSECTION_RELATED_I"
]

df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Define target column
target = "SEVERITY"

# Define numerical columns
num_cols = [
    "NUM_UNITS",
    "INJURIES_TOTAL",
    "INJURIES_FATAL",
    "CRASH_HOUR",
    "CRASH_DAY_OF_WEEK",
    "CRASH_MONTH"
]

# Keep only numerical columns that actually exist
num_cols = [col for col in num_cols if col in df.columns]

# Define categorical columns
cat_cols = [
    col for col in df.columns
    if col != target and col not in num_cols
]

print("Numerical Columns:")
print(num_cols)

print("\nCategorical Columns:")
print(cat_cols)

Numerical Columns:
['NUM_UNITS', 'INJURIES_TOTAL', 'INJURIES_FATAL', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH']

Categorical Columns:
['TRAFFIC_CONTROL_DEVICE', 'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE', 'ALIGNMENT', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'CRASH_TYPE', 'DAMAGE', 'PRIM_CONTRIBUTORY_CAUSE']


### Handle Missing Values

In [4]:
# Fill numerical missing values with median
for col in num_cols:
    if df[col].isnull().all():
        df[col] = df[col].fillna(0)
    else:
        df[col] = df[col].fillna(df[col].median())

# Fill categorical and target missing values with mode
for col in cat_cols + [target]:
    if df[col].isnull().all():
        df[col] = df[col].fillna("UNKNOWN")
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

### Balance the dataset

In [5]:
# Separate classes
df_low = df[df[target] == "LOW"]
df_medium = df[df[target] == "MEDIUM"]
df_high = df[df[target] == "HIGH"]

# Undersample LOW severity class
df_low_under = df_low.sample(
    n=min(200000, len(df_low)),
    random_state=42
)

# Recombine dataset
df_balanced = pd.concat(
    [df_low_under, df_medium, df_high],
    axis=0
)

# Shuffle dataset
df_balanced = df_balanced.sample(
    frac=1,
    random_state=1
).reset_index(drop=True)

print(df_balanced[target].value_counts())

SEVERITY
LOW       200000
MEDIUM     83111
HIGH       17932
Name: count, dtype: int64


### Train / Validation / Test Split

In [6]:
# 70% train, 30% temporary
train_df, temp_df = train_test_split(
    df_balanced,
    test_size=0.30,
    random_state=1,
    stratify=df_balanced[target]
)

# Split temporary set into 20% validation and 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.3333,
    random_state=1,
    stratify=temp_df[target]
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (210730, 18)
Validation shape: (60211, 18)
Test shape: (30102, 18)


### Encoding and Scaling

In [7]:
scaler = MinMaxScaler()
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

def transform_features(data, is_train=False):
    if is_train:
        num_part = scaler.fit_transform(data[num_cols])
        cat_part = encoder.fit_transform(data[cat_cols])
    else:
        num_part = scaler.transform(data[num_cols])
        cat_part = encoder.transform(data[cat_cols])

    return np.hstack([num_part, cat_part])


# Transform features
X_train = transform_features(train_df, is_train=True)
X_val = transform_features(val_df)
X_test = transform_features(test_df)

# Target labels
y_train = train_df[target]
y_val = val_df[target]
y_test = test_df[target]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (210730, 146)
X_val shape: (60211, 146)
X_test shape: (30102, 146)


### Reusable Evaluation Function

In [8]:
def evaluate_model(model_name, y_true, y_pred):
    print(f"\n--- {model_name} Performance ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision (Macro): {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"Recall (Macro): {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1 Score (Macro): {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

### Logistic Regression

In [9]:
log_reg_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)

log_reg_model.fit(X_train, y_train)

log_val_pred = log_reg_model.predict(X_val)
log_test_pred = log_reg_model.predict(X_test)

evaluate_model("Logistic Regression Validation", y_val, log_val_pred)
evaluate_model("Logistic Regression Test", y_test, log_test_pred)


--- Logistic Regression Validation Performance ---
Accuracy: 0.8447
Precision (Macro): 0.6665
Recall (Macro): 0.7084
F1 Score (Macro): 0.6693

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.24      0.51      0.32      3587
         LOW       1.00      0.95      0.97     40001
      MEDIUM       0.76      0.66      0.71     16623

    accuracy                           0.84     60211
   macro avg       0.67      0.71      0.67     60211
weighted avg       0.89      0.84      0.86     60211


--- Logistic Regression Test Performance ---
Accuracy: 0.8462
Precision (Macro): 0.6699
Recall (Macro): 0.7140
F1 Score (Macro): 0.6732

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.24      0.52      0.33      1793
         LOW       1.00      0.95      0.97     19999
      MEDIUM       0.77      0.67      0.71      8310

    accuracy                           0.85     30102
   macro avg       

### Random Forest

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_val_pred = rf_model.predict(X_val)
rf_test_pred = rf_model.predict(X_test)

evaluate_model("Random Forest Validation", y_val, rf_val_pred)
evaluate_model("Random Forest Test", y_test, rf_test_pred)


--- Random Forest Validation Performance ---
Accuracy: 0.9059
Precision (Macro): 0.7610
Recall (Macro): 0.6708
F1 Score (Macro): 0.6578

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.53      0.09      0.15      3587
         LOW       1.00      0.95      0.97     40001
      MEDIUM       0.76      0.98      0.85     16623

    accuracy                           0.91     60211
   macro avg       0.76      0.67      0.66     60211
weighted avg       0.90      0.91      0.89     60211


--- Random Forest Test Performance ---
Accuracy: 0.9049
Precision (Macro): 0.7519
Recall (Macro): 0.6694
F1 Score (Macro): 0.6560

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.50      0.08      0.15      1793
         LOW       0.99      0.95      0.97     19999
      MEDIUM       0.76      0.97      0.85      8310

    accuracy                           0.90     30102
   macro avg       0.75      0.

### Xgboost

In [11]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train_encoded)

xgb_val_pred_encoded = xgb_model.predict(X_val)
xgb_test_pred_encoded = xgb_model.predict(X_test)

xgb_val_pred = label_encoder.inverse_transform(xgb_val_pred_encoded)
xgb_test_pred = label_encoder.inverse_transform(xgb_test_pred_encoded)

evaluate_model("XGBoost Validation", y_val, xgb_val_pred)
evaluate_model("XGBoost Test", y_test, xgb_test_pred)


--- XGBoost Validation Performance ---
Accuracy: 0.9104
Precision (Macro): 0.9006
Recall (Macro): 0.6709
F1 Score (Macro): 0.6515

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.95      0.06      0.12      3587
         LOW       1.00      0.95      0.97     40001
      MEDIUM       0.76      1.00      0.86     16623

    accuracy                           0.91     60211
   macro avg       0.90      0.67      0.65     60211
weighted avg       0.93      0.91      0.89     60211


--- XGBoost Test Performance ---
Accuracy: 0.9101
Precision (Macro): 0.9038
Recall (Macro): 0.6701
F1 Score (Macro): 0.6500

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.96      0.06      0.12      1793
         LOW       1.00      0.95      0.97     19999
      MEDIUM       0.75      1.00      0.86      8310

    accuracy                           0.91     30102
   macro avg       0.90      0.67      0.65

### LightGBM

In [12]:
lgbm_model = LGBMClassifier(
    objective="multiclass",
    num_class=len(label_encoder.classes_),
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(X_train, y_train_encoded)

lgbm_val_pred_encoded = lgbm_model.predict(X_val)
lgbm_test_pred_encoded = lgbm_model.predict(X_test)

lgbm_val_pred = label_encoder.inverse_transform(lgbm_val_pred_encoded)
lgbm_test_pred = label_encoder.inverse_transform(lgbm_test_pred_encoded)

evaluate_model("LightGBM Validation", y_val, lgbm_val_pred)
evaluate_model("LightGBM Test", y_test, lgbm_test_pred)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014939 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 351
[LightGBM] [Info] Number of data points in the train set: 210730, number of used features: 143
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


/opt/anaconda3/envs/DataLab/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/DataLab/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- LightGBM Validation Performance ---
Accuracy: 0.8527
Precision (Macro): 0.6691
Recall (Macro): 0.7078
F1 Score (Macro): 0.6756

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.24      0.47      0.32      3587
         LOW       1.00      0.95      0.97     40001
      MEDIUM       0.76      0.70      0.73     16623

    accuracy                           0.85     60211
   macro avg       0.67      0.71      0.68     60211
weighted avg       0.89      0.85      0.87     60211


--- LightGBM Test Performance ---
Accuracy: 0.8549
Precision (Macro): 0.6732
Recall (Macro): 0.7143
F1 Score (Macro): 0.6807

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.25      0.49      0.33      1793
         LOW       1.00      0.95      0.97     19999
      MEDIUM       0.77      0.71      0.74      8310

    accuracy                           0.85     30102
   macro avg       0.67      0.71      0.